# IEEE Xplore foundation/agentic robotic-hand review

This notebook analyzes **all deduplicated records** returned by the crawler. It does not exclude uncited papers or papers without IEEE index terms—both exclusions would systematically hide recent state-of-the-art work.

The concept and evidence labels below are transparent title/abstract/keyword heuristics for exploration and screening priority. They are not automatic inclusion decisions.

In [ ]:
from collections import Counter
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
pd.set_option("display.max_colwidth", 100)

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "crawlers/IEEE-Xplore-API_p3"]
crawler_dir = next((path for path in candidates if (path / "main.py").is_file()), None)
if crawler_dir is None:
    raise FileNotFoundError("Run this notebook from the repository root or IEEE-Xplore-API_p3 directory.")

data_dir = crawler_dir / "data_foundation_agentic"
files = sorted(data_dir.glob("data_*.json"), key=lambda path: int(path.stem.split("_")[-1]))
if not files:
    raise FileNotFoundError(f"No IEEE result pages found in {data_dir}")

raw_articles = []
for path in files:
    with path.open(encoding="utf-8") as file:
        raw_articles.extend(json.load(file).get("articles", []))

articles_by_id = {}
for article in raw_articles:
    identifier = article.get("article_number") or article.get("doi") or article.get("title")
    articles_by_id.setdefault(str(identifier), article)
articles = list(articles_by_id.values())
df = pd.DataFrame(articles).copy()

required_columns = [
    "article_number", "doi", "title", "abstract", "publication_year",
    "publication_title", "content_type", "access_type",
    "citing_paper_count", "download_count", "html_url",
]
for column in required_columns:
    if column not in df:
        df[column] = ""

def index_term_text(article):
    chunks = []
    groups = article.get("index_terms") or {}
    if isinstance(groups, dict):
        for group in groups.values():
            if isinstance(group, dict):
                chunks.extend(str(term) for term in group.get("terms", []) or [])
    return " ".join(chunks)

df["index_term_text"] = [index_term_text(article) for article in articles]
df["search_text"] = (
    df["title"].fillna("").astype(str) + " "
    + df["abstract"].fillna("").astype(str) + " "
    + df["index_term_text"]
).str.lower()
df["year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")
df["citations"] = pd.to_numeric(df["citing_paper_count"], errors="coerce").fillna(0)
df["downloads"] = pd.to_numeric(df["download_count"], errors="coerce").fillna(0)
df["content_group"] = df["content_type"].fillna("Unknown").replace("", "Unknown")
df["access_group"] = df["access_type"].fillna("Unknown").replace("", "Unknown")

print(f"Pages: {len(files)}")
print(f"Raw records: {len(raw_articles):,}")
print(f"Unique records: {len(df):,}")
print(f"Publication years: {df['year'].min()}–{df['year'].max()}")

In [ ]:
# Search concepts and contribution/evidence signals visible in returned metadata.
concept_terms = {
    "Hand embodiment": [
        "robotic hand", "robot hand", "dexterous hand", "anthropomorphic hand",
        "artificial hand", "prosthetic hand", "bionic hand", "multifingered hand",
        "multi-fingered hand", "multifinger hand", "five-finger hand", "robotic palm",
        "robotic finger", "soft robotic hand", "tendon-driven hand", "hand morphology",
        "finger synergy", "hand synergy", "dexterous grasp", "multifinger grasp",
    ],
    "Manipulation": [
        "grasping", "grasp generation", "functional grasp", "task-oriented grasp",
        "dexterous manipulation", "in-hand manipulation", "in hand manipulation",
        "object manipulation", "hand-object interaction", "finger control", "hand control",
    ],
    "Foundation / multimodal": [
        "foundation model", "large language model", "vision-language model",
        "visual language model", "vision-language-action", "vision language action",
        "multimodal large language model", "generalist policy", "vision-language policy",
        "promptable foundation model", "tactile-language-action", "latent action representation",
    ],
    "Agentic reasoning": [
        "agentic ai", "llm planner", "vlm planner", "multimodal planner",
        "language-guided planning", "open-world planning", "task decomposition",
        "chain-of-thought", "self-evaluation", "self-reflection", "closed-loop reasoning",
        "autonomous reasoning", "reasoning-based manipulation", "failure recovery",
    ],
}

method_terms = {
    "Foundation model": ["foundation model"],
    "LLM": ["large language model", " llm ", "llm-based", "llm planner"],
    "VLM": ["vision-language model", "visual language model", " vlm ", "vlm-based"],
    "VLA": ["vision-language-action", "vision language action", " vla ", "vla-based"],
    "Planning / reasoning": ["task planning", "chain-of-thought", "reasoning", "task decomposition"],
    "Open vocabulary / zero shot": ["open-vocabulary", "open vocabulary", "zero-shot", "zero shot"],
    "Tactile multimodality": ["tactile-language", "tactile sensing", "visuo-tactile", "vision-tactile"],
}

evidence_terms = {
    "Real hardware": ["real-world", "real world", "real robot", "physical robot", "physical hand"],
    "Generalization": ["generalization", "generalizability", "unseen object", "novel object", "open-world"],
    "Benchmark / dataset": ["benchmark", "dataset", "data set"],
    "Simulation-to-real": ["sim-to-real", "sim2real", "simulation-to-real"],
    "Open code / materials": ["open-source", "open source", "code and data", "project website"],
}

def matching_terms(text, terms):
    padded = f" {text} "
    return [term.strip() for term in terms if term in padded]

concept_columns = {}
for concept, terms in concept_terms.items():
    column = re.sub(r"[^a-z0-9]+", "_", concept.lower()).strip("_")
    concept_columns[concept] = column
    df[f"{column}_terms"] = df["search_text"].map(lambda text, terms=terms: matching_terms(text, terms))
    df[column] = df[f"{column}_terms"].map(bool)

method_columns = {}
for method, terms in method_terms.items():
    column = re.sub(r"[^a-z0-9]+", "_", method.lower()).strip("_")
    method_columns[method] = column
    df[column] = df["search_text"].map(lambda text, terms=terms: bool(matching_terms(text, terms)))

for evidence, terms in evidence_terms.items():
    column = re.sub(r"[^a-z0-9]+", "_", evidence.lower()).strip("_")
    df[column] = df["search_text"].map(lambda text, terms=terms: bool(matching_terms(text, terms)))

df["ai_visible"] = df["foundation_multimodal"] | df["agentic_reasoning"]
df["scope_score"] = (df["hand_embodiment"].astype(int) + df["manipulation"].astype(int) + df["ai_visible"].astype(int))
evidence_columns = [re.sub(r"[^a-z0-9]+", "_", key.lower()).strip("_") for key in evidence_terms]
df["evidence_score"] = df[evidence_columns].sum(axis=1)
df["screening_tier"] = np.select(
    [df["scope_score"].eq(3), df["scope_score"].eq(2)],
    ["All three concepts visible", "Two concepts visible"],
    default="Full-text metadata gap",
)

## Corpus shape and temporal shift

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)

year_counts = df.dropna(subset=["year"])["year"].astype(int).value_counts().sort_index()
axes[0, 0].bar(year_counts.index, year_counts.values, color="#4C78A8")
axes[0, 0].plot(year_counts.index, year_counts.values, color="#1F3B5C", marker="o")
axes[0, 0].set(title="Search results accelerated sharply after 2023", xlabel="Publication year", ylabel="Papers")
axes[0, 0].set_xticks(year_counts.index)

yearly = df.dropna(subset=["year"]).assign(year=lambda x: x["year"].astype(int)).groupby("year")
yearly_shares = pd.DataFrame({
    "Hand embodiment": yearly["hand_embodiment"].mean(),
    "Foundation / multimodal": yearly["foundation_multimodal"].mean(),
    "Agentic reasoning": yearly["agentic_reasoning"].mean(),
}) * 100
yearly_shares.plot(ax=axes[0, 1], marker="o", linewidth=2.2)
axes[0, 1].set(title="Concept prevalence within each year", xlabel="Publication year", ylabel="Share of papers (%)")
axes[0, 1].set_xticks(yearly_shares.index)
axes[0, 1].legend(frameon=False)

df["content_group"].value_counts().sort_values().plot.barh(ax=axes[1, 0], color="#72B7B2")
axes[1, 0].set(title="Document types", xlabel="Papers", ylabel="")

tier_order = ["All three concepts visible", "Two concepts visible", "Full-text metadata gap"]
tier_counts = df["screening_tier"].value_counts().reindex(tier_order, fill_value=0)
tier_counts.plot.bar(ax=axes[1, 1], color=["#54A24B", "#F2CF5B", "#E45756"])
axes[1, 1].set(title="How much of the API match is visible in metadata?", xlabel="", ylabel="Papers")
axes[1, 1].tick_params(axis="x", rotation=15)

fig.suptitle("Foundation/agentic robotic-hand search: corpus overview", fontsize=16, fontweight="bold")
plt.show()

## What the query is actually retrieving

In [ ]:
# Exact concept intersections expose broad-query leakage better than isolated totals.
intersection_columns = ["hand_embodiment", "manipulation", "foundation_multimodal", "agentic_reasoning"]
intersection_names = ["Hand", "Manipulation", "Foundation", "Agentic"]
patterns = df[intersection_columns].value_counts().reset_index(name="papers")
patterns["intersection"] = patterns.apply(
    lambda row: " + ".join(name for name, column in zip(intersection_names, intersection_columns) if row[column]) or "No concept visible",
    axis=1,
)
patterns = patterns.sort_values("papers").tail(12)

binary = df[intersection_columns].astype(int)
jaccard = pd.DataFrame(index=intersection_names, columns=intersection_names, dtype=float)
for i, left in enumerate(intersection_columns):
    for j, right in enumerate(intersection_columns):
        union = (binary[left] | binary[right]).sum()
        jaccard.iloc[i, j] = (binary[left] & binary[right]).sum() / union if union else 0

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
axes[0].barh(patterns["intersection"], patterns["papers"], color="#B279A2")
axes[0].set(title="Largest exact concept intersections", xlabel="Papers", ylabel="")

image = axes[1].imshow(jaccard, vmin=0, vmax=1, cmap="YlGnBu")
axes[1].set_title("Concept co-occurrence (Jaccard similarity)")
axes[1].set_xticks(range(len(intersection_names)), intersection_names, rotation=25, ha="right")
axes[1].set_yticks(range(len(intersection_names)), intersection_names)
for row in range(len(intersection_names)):
    for col in range(len(intersection_names)):
        axes[1].text(col, row, f"{jaccard.iloc[row, col]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=axes[1], label="Jaccard similarity", shrink=0.8)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

ax.barh(patterns["intersection"], patterns["papers"], color="#B279A2")
ax.set(
    title="Largest exact concept intersections",
    xlabel="Papers",
    ylabel="",
)

plt.show()

## Method evolution and terminology

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

image = ax.imshow(year_methods.T, aspect="auto", cmap="Blues")
ax.set(
    title="Method families by year",
    xlabel="Publication year",
    ylabel="Method family",
)
ax.set_xticks(range(len(year_methods.index)), year_methods.index)
ax.set_yticks(range(len(year_methods.columns)), year_methods.columns)

for row in range(year_methods.shape[1]):
    for col in range(year_methods.shape[0]):
        ax.text(
            col, row, int(year_methods.iloc[col, row]),
            ha="center", va="center", fontsize=9
        )

fig.colorbar(image, ax=ax, label="Papers", shrink=0.8)
plt.show()


In [ ]:
year_methods = (
    df.dropna(subset=["year"])
    .assign(year=lambda frame: frame["year"].astype(int))
    .groupby("year")[[*method_columns.values()]]
    .sum()
    .rename(columns={value: key for key, value in method_columns.items()})
)

phrase_counts = Counter()
for concept in concept_terms:
    column = f"{concept_columns[concept]}_terms"
    for matches in df[column]:
        phrase_counts.update(matches)
top_phrases = pd.Series(dict(phrase_counts.most_common(20))).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(17, 7), constrained_layout=True)
image = axes[0].imshow(year_methods.T, aspect="auto", cmap="Blues")
axes[0].set(title="Method families by year", xlabel="Publication year", ylabel="Method family")
axes[0].set_xticks(range(len(year_methods.index)), year_methods.index)
axes[0].set_yticks(range(len(year_methods.columns)), year_methods.columns)
for row in range(year_methods.shape[1]):
    for col in range(year_methods.shape[0]):
        axes[0].text(col, row, int(year_methods.iloc[col, row]), ha="center", va="center", fontsize=9)
fig.colorbar(image, ax=axes[0], label="Papers", shrink=0.8)

top_phrases.plot.barh(ax=axes[1], color="#4C78A8")
axes[1].set(title="Most visible query phrases", xlabel="Papers containing phrase", ylabel="")
plt.show()

## Recent SOTA signals—without using citations as an inclusion rule

In [ ]:
# Recency, visible scope, and experimental evidence provide a screening aid; the score is not a quality claim.
min_year = int(df["year"].dropna().min())
df["recency_score"] = (df["year"].fillna(min_year).astype(int) - min_year).clip(lower=0)
df["sota_screen_score"] = 2 * df["scope_score"] + df["evidence_score"] + df["recency_score"]
recent = df[df["year"].ge(2023)].copy()
recent["year_jitter"] = recent["year"].astype(float) + np.random.default_rng(7).normal(0, 0.06, len(recent))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
scatter = axes[0].scatter(
    recent["year_jitter"], recent["evidence_score"],
    s=25 + 18 * np.log1p(recent["downloads"]),
    c=recent["scope_score"], cmap="viridis", vmin=0, vmax=3,
    alpha=0.65, edgecolor="white", linewidth=0.4,
)
axes[0].set(title="Recent papers: visible scope vs. evidence signals", xlabel="Publication year", ylabel="Evidence signals in metadata")
axes[0].set_xticks(sorted(recent["year"].dropna().astype(int).unique()))
fig.colorbar(scatter, ax=axes[0], label="Visible scope concepts (0–3)", shrink=0.8)

evidence_counts = df[evidence_columns].sum().rename(index={column: label for column, label in zip(evidence_columns, evidence_terms)})
evidence_counts.sort_values().plot.barh(ax=axes[1], color="#54A24B")
axes[1].set(title="Experimental and dissemination signals", xlabel="Papers", ylabel="")
plt.show()

candidate_columns = ["year", "title", "scope_score", "evidence_score", "citations", "downloads", "html_url"]
display(
    df.sort_values(["sota_screen_score", "year", "downloads"], ascending=False)
    [candidate_columns]
    .head(30)
    .reset_index(drop=True)
)

## Venue concentration, attention, and metadata quality

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)

venues = df["publication_title"].fillna("Unknown").replace("", "Unknown")
venue_counts = venues.value_counts()
venue_counts.head(15).sort_values().plot.barh(ax=axes[0, 0], color="#F58518")
axes[0, 0].set(title="Top publication venues", xlabel="Papers", ylabel="")

cumulative = venue_counts.cumsum() / venue_counts.sum() * 100
axes[0, 1].plot(range(1, len(cumulative) + 1), cumulative.values, linewidth=2.5)
axes[0, 1].axhline(80, color="#E45756", linestyle="--", label="80% of papers")
axes[0, 1].set(title="Venue concentration curve", xlabel="Venues ranked by paper count", ylabel="Cumulative share of papers (%)")
axes[0, 1].legend(frameon=False)

impact = axes[1, 0].scatter(
    df["downloads"] + 1, df["citations"] + 1,
    c=df["year"].fillna(df["year"].min()),
    s=35 + 25 * df["scope_score"], cmap="plasma", alpha=0.65, edgecolor="white", linewidth=0.4,
)
axes[1, 0].set_xscale("log")
axes[1, 0].set_yscale("log")
axes[1, 0].set(title="Downloads and citations are context, not eligibility", xlabel="Downloads + 1 (log scale)", ylabel="Citations + 1 (log scale)")
fig.colorbar(impact, ax=axes[1, 0], label="Publication year", shrink=0.8)

quality = pd.Series({
    "Abstract missing": df["abstract"].fillna("").str.strip().eq("").sum(),
    "Index terms missing": df["index_term_text"].str.strip().eq("").sum(),
    "Zero citations": df["citations"].eq(0).sum(),
    "AI phrase not visible": (~df["ai_visible"]).sum(),
    "Hand/task phrase not visible": (~(df["hand_embodiment"] | df["manipulation"])).sum(),
}).sort_values()
quality.plot.barh(ax=axes[1, 1], color="#E45756")
axes[1, 1].set(title="Metadata and query-visibility gaps", xlabel="Papers", ylabel="")
for patch in axes[1, 1].patches:
    axes[1, 1].text(patch.get_width() + 1, patch.get_y() + patch.get_height() / 2, int(patch.get_width()), va="center")

plt.show()